# Tarea 2 — Reconstrucción 3D con COLMAP
**IIC3912 Tópicos Avanzados de Gráfica Computacional**

Este notebook es el punto de partida común para todos los experimentos.
Ejecuta las celdas en orden para instalar dependencias, montar Drive y descargar el dataset.

In [1]:
# Instalación de dependencias
#!pip install pycolmap plotly open3d pandas tqdm -q

import os, shutil, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pycolmap
from pathlib import Path
from PIL import Image as PILImage

print('pycolmap:', pycolmap.__version__)

pycolmap: 4.0.4


In [2]:
SCENES = ['garden', 'bicycle', 'bonsai', 'counter']

IMG_SCALE = 4  # resolución: 1 (full), 2, 4, 8  — se recomienda 4 en Colab CPU

In [4]:
# ── Google Drive y paths ───────────────────────────────────────────────────
#from google.colab import drive
#drive.mount('/content/drive')

DRIVE_BASE = Path('./tarea2_mipnerf_dataset')
BASE       = Path('./colmap_work')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

def scene_paths(scene):
    return {
        'drive'   : DRIVE_BASE / scene,
        'images'  : BASE / scene / 'images',
        'sparse'  : BASE / scene / 'sparse',
        'db'      : BASE / scene / 'colmap.db',
    }

# Crear directorios de trabajo para todas las escenas
for scene in SCENES:
    p = scene_paths(scene)
    for d in [p['images'], p['sparse'] / 'student', p['sparse'] / 'official']:
        d.mkdir(parents=True, exist_ok=True)

In [5]:
# ── Descarga y extracción del dataset ─────────────────────────────────────
# El zip (~1.8 GB) se descarga una sola vez a Drive.
# Las sesiones siguientes lo reutilizan directamente.
import zipfile

DATASET_URL = 'https://storage.googleapis.com/gresearch/refraw360/360_v2.zip'
ZIP_PATH    = DRIVE_BASE / '360_v2.zip'

def download_and_extract(scene):
    dst = DRIVE_BASE / scene
    if dst.exists():
        print(f'[{scene}] ya en Drive')
        return
    if not ZIP_PATH.exists():
        print('Descargando dataset (~1.8 GB)...')
        ret = os.system(f'wget -q --show-progress -O "{ZIP_PATH}" "{DATASET_URL}"')
        if ret != 0:
            raise RuntimeError('Descarga fallida. Sube el zip manualmente a Drive.')
    print(f'[{scene}] extrayendo...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        archivos_escena = [m for m in zip_ref.namelist() if m.startswith(f"{scene}/")]
        zip_ref.extractall(path=DRIVE_BASE, members=archivos_escena)
    if not dst.exists(): #ret != 0 or not dst.exists():
        raise RuntimeError(f'Error extrayendo {scene}')
    print(f'[{scene}] listo en {dst}')

def prepare_images(scene):
    paths = scene_paths(scene)
    dst   = paths['images']
    existing = list(dst.glob('*.jpg')) + list(dst.glob('*.JPG')) + list(dst.glob('*.png'))
    if existing:
        print(f'[{scene}] {len(existing)} imágenes ya en local')
        return
    scale_folder = f'images_{IMG_SCALE}' if IMG_SCALE > 1 else 'images'
    src = paths['drive'] / scale_folder
    if not src.exists():
        src = paths['drive'] / 'images'
        print(f'[{scene}] {scale_folder}/ no encontrada → usando images/')
    imgs = sorted(list(src.glob('*.jpg')) + list(src.glob('*.JPG')) + list(src.glob('*.png')))
    print(f'[{scene}] copiando {len(imgs)} imágenes desde {src.name}...')
    for img_path in imgs:
        shutil.copy(str(img_path), str(dst / img_path.name))
    with PILImage.open(list(dst.glob('*'))[0]) as im:
        print(f'[{scene}] resolución: {im.size} | total: {len(imgs)} imgs')

# Descargar y preparar las 4 escenas
for scene in SCENES:
    download_and_extract(scene)
    prepare_images(scene)

[garden] ya en Drive
[garden] 185 imágenes ya en local
[bicycle] ya en Drive
[bicycle] 194 imágenes ya en local
[bonsai] ya en Drive
[bonsai] 292 imágenes ya en local
[counter] ya en Drive
[counter] 240 imágenes ya en local


In [6]:
fractions = [0.15, 0.3, 0.45, 0.6] # La idea inicial era hacer cada 25% hasta llegar al 100%,
                                    # computador no lograba pasar más del 70%
SEED = 42

resultados_expB = []

In [7]:
extraction_opts = pycolmap.FeatureExtractionOptions()
extraction_opts.max_image_size = 1024
extraction_opts.num_threads = 4

for scene in SCENES:
    paths = scene_paths(scene)
    img_dir = paths['images']
    
    all_images = sorted([f.name for f in img_dir.glob('*.JPG')])
    
    for frac in fractions:
        print(f"\nProcesando {scene} | {frac*100}%")
        random.seed(SEED)
        num_img = int(len(all_images) * frac)
        subset_images = random.sample(all_images, num_img) #Seleccion imagenes random
        
        frac_str = f"frac_{int(frac*100)}"
        db_path = BASE / scene / f"colmap_{frac_str}.db"
        sparse_out = paths['sparse'] / 'student' / frac_str
        sparse_out.mkdir(parents=True, exist_ok=True)
        
        if db_path.exists():
            db_path.unlink()
            
        start_time = time.time()
        
        # SfM: extraccion y matching
        pycolmap.extract_features(db_path, img_dir, image_names=subset_images, 
                                  extraction_options=extraction_opts)
        pycolmap.match_exhaustive(db_path)
        
        # Guardamos mapeo
        maps = pycolmap.incremental_mapping(db_path, img_dir, sparse_out)
        
        exec_time = time.time() - start_time
        
        num_components = len(maps) if maps else 0  
        reg_images, num_points, mean_error = 0, 0, 0.0
        track_length_prom = 0.0
        
        if num_components > 0:
            # Selec mejor sub-modelo
            best_id, best_rec = max(maps.items(), key=lambda kv: kv[1].num_reg_images())
            
            # Det total imagenes y registradas
            reg_images = len(best_rec.images)
            num_points = len(best_rec.points3D)

            mean_error = best_rec.compute_mean_reprojection_error()
            
            # Cálculo de metrica propuesta
            if num_points > 0:
                # Det total de camaras que observan cada punto y calculamos promedio
                total_observations = sum(len(p.track.elements) for p in best_rec.points3D.values())
                track_length_prom = total_observations / num_points

        resultados_expB.append({
            'Escena': scene,
            'Fraccion': frac,
            'a) Imgs tot': num_img,
            'a) Imgs reg': reg_images,
            'a) % Registrado': (reg_images / num_img * 100) if num_img > 0 else 0,
            'b) Puntos 3D': num_points,
            'c) Error reproy': mean_error,
            'd) Componentes': num_components,
            'e) Tiempo': exec_time,
            'f) Track length prom': track_length_prom
        })

df_resultados = pd.DataFrame(resultados_expB)
display(df_resultados)


Procesando garden | 15.0%


I20260514 03:26:35.293253 140154184398528 feature_extraction.cc:506] === Feature extraction ===
I20260514 03:26:35.294104 140154142435008 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 03:26:35.294545 140154134042304 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 03:26:35.294618 140154125649600 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 03:26:35.295894 140153909671616 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 03:26:38.783906 140153901278912 feature_extraction.cc:284] Processed file [1/27]
I20260514 03:26:38.783961 140153901278912 feature_extraction.cc:287]   Name:            DSC07964.JPG
I20260514 03:26:38.783965 140153901278912 feature_extraction.cc:296]   Dimensions:      1297 x 840
I20260514 03:26:38.783968 140153901278912 feature_extraction.cc:299]   Camera:          #3 - SIMPLE_RADIAL
I20260514 03:26:38.783973 140153901278912 feature_extraction.cc:302]   Focal Length:    1556.40px
I20260514 03:26:38.783989 140153901278912 fe


Procesando garden | 30.0%


I20260514 03:27:50.501176 140154134042304 feature_extraction.cc:506] === Feature extraction ===
I20260514 03:27:50.502823 140154167613120 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 03:27:50.503014 140154159220416 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 03:27:50.503202 140154125649600 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 03:27:50.507066 140153909671616 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 03:27:53.970133 140153901278912 feature_extraction.cc:284] Processed file [1/55]
I20260514 03:27:53.970274 140153901278912 feature_extraction.cc:287]   Name:            DSC07957.JPG
I20260514 03:27:53.970283 140153901278912 feature_extraction.cc:296]   Dimensions:      1297 x 840
I20260514 03:27:53.970287 140153901278912 feature_extraction.cc:299]   Camera:          #1 - SIMPLE_RADIAL
I20260514 03:27:53.970293 140153901278912 feature_extraction.cc:302]   Focal Length:    1556.40px
I20260514 03:27:53.970331 140153901278912 fe


Procesando garden | 45.0%


...


Procesando counter | 45.0%


I20260514 05:18:39.727818 140153901278912 feature_extraction.cc:284] Processed file [1/108]
I20260514 05:18:39.727896 140153901278912 feature_extraction.cc:287]   Name:            DSCF5865.JPG
I20260514 05:18:39.727901 140153901278912 feature_extraction.cc:296]   Dimensions:      779 x 519
I20260514 05:18:39.727904 140153901278912 feature_extraction.cc:299]   Camera:          #4 - SIMPLE_RADIAL
I20260514 05:18:39.727908 140153901278912 feature_extraction.cc:302]   Focal Length:    934.80px
I20260514 05:18:39.727918 140153901278912 feature_extraction.cc:306]   Features:        2939 (SIFT)
I20260514 05:18:39.785206 140153901278912 feature_extraction.cc:284] Processed file [2/108]
I20260514 05:18:39.785274 140153901278912 feature_extraction.cc:287]   Name:            DSCF5858.JPG
I20260514 05:18:39.785280 140153901278912 feature_extraction.cc:296]   Dimensions:      779 x 519
I20260514 05:18:39.785283 140153901278912 feature_extraction.cc:299]   Camera:          #1 - SIMPLE_RADIAL
I202605


Procesando counter | 60.0%


I20260514 05:22:20.537413 140154134042304 feature_extraction.cc:506] === Feature extraction ===
I20260514 05:22:20.539342 140154167613120 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 05:22:20.539605 140154159220416 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 05:22:20.539979 140154125649600 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 05:22:20.540814 140153909671616 sift.cc:763] Creating SIFT CPU feature extractor
I20260514 05:22:21.888736 140153901278912 feature_extraction.cc:284] Processed file [1/144]
I20260514 05:22:21.888827 140153901278912 feature_extraction.cc:287]   Name:            DSCF5865.JPG
I20260514 05:22:21.888838 140153901278912 feature_extraction.cc:296]   Dimensions:      779 x 519
I20260514 05:22:21.888842 140153901278912 feature_extraction.cc:299]   Camera:          #4 - SIMPLE_RADIAL
I20260514 05:22:21.888847 140153901278912 feature_extraction.cc:302]   Focal Length:    934.80px
I20260514 05:22:21.888862 140153901278912 fea

,Escena,Fraccion,a) Imgs tot,a) Imgs reg,a) % Registrado,b) Puntos 3D,c) Error reproy,d) Componentes,e) Tiempo,f) Track length prom
0,garden,0.15,27,27,100.000000,7612,0.337816,1,75.377213,3.739359
1,garden,0.30,55,55,100.000000,29050,0.379404,1,263.902234,4.511979
2,garden,0.45,83,83,100.000000,49410,0.406595,1,575.989415,5.453410
3,garden,0.60,111,111,100.000000,67858,0.424190,1,977.989378,6.157682
4,bicycle,0.15,29,11,37.931034,790,0.511937,2,91.629532,3.774684
5,bicycle,0.30,58,34,58.620690,3859,0.404044,2,318.091786,3.909303
6,bicycle,0.45,87,86,98.850575,14506,0.436916,1,659.868517,4.205570
7,bicycle,0.60,116,115,99.137931,24179,0.461434,1,1199.451301,4.506018
8,bonsai,0.15,43,43,100.000000,19926,0.223486,1,95.699596,3.959952
9,bonsai,0.30,87,87,100.000000,39641,0.281445,1,282.654917,5.146414


Justificación métrica propuesta:
Propusimos la métrica Track Lenght Promedio. Que consiste en determinar desde cuantas cámaras es visto cada punto y determinar el promedio general. Decidimos usar esta métrica porque, dado que una parte importante de SfM son los puntos, a partir de los cuales, al tringularlos, se genera la nube de puntos, y sabemos que mientras más cámaras hayan visto este punto, es más robusto y claro, mayor calidad. Por lo que saber esta métrica, nos entrega un nuevo índice de calidad de la reconstrucción.

Discusión:
Considerando los valores obtenidos con cada metrica, de cada escena, comparando los porcentajes de una misma escena, y entre escenas, podemos determinar el funcionamiento  de SfM. Logramos observar por cada métrica:
- Imágenes registradas y porcentaje sobre total: Mirando esta métrica en bycicle, vemos que el porcentaje de registro es considerablemente bajo (37.9%) cuando la fracción de imagenes es baja, lo cual indicaría una falta de overlap. Pero sabemos que para estimar la pose relativa de una camara usando PnP, es necesario que hayan suficientes correspondencias, por lo que tiene sentido que cuando hay pocas fotos, haya menor correspondencia dado los salto visuales muy grande, impidiendo que la cámara no se localice. Pero por otro lado, vemos que con garden y bonsai esta métrica se comporta bastante bien, el porcentaje de registradas es igual en cada fracción (100%), indicando escenas mucho más densa y ricas en texturas.

- N° puntos 3D reconstruidos: Esta métrica se comporta de forma similar en cada escena. La cantidad de puntos aumenta considerablemente entre cada fracción, aumenta justamente el doble, o en casos, hasta el cuadruple, como en garden, donde con fracción de 0.15 tiene 7612 puntos, y luego en la 0.3 tiene 29050 puntos. Esto tiene mucha relación con lo mencionado en la métrica anterior, menos imagenes implican menor triangulación, y por ende, menor reconstrucción de punto 3D.

- Error de reproyección promedio: Aquí, de igual forma, se comporta de igual manera entre escenas, y curiosamente vemos que a medida que la fracción es mayor, el error crece, con counter vemos que es caso mayor, donde al 0.15 el error es 0.29 y luego al 0.6 es de 0.4. Esto se podría relacionar con el bundle adjustment, ya que a menor cantidad de imagenes hay menos puntos que optimizar para ajustar las poses de las camaras, y al agregar más imagenes es más trabajo.

- Componentes conexas: se ve que nuevamente se comporta bien con garden y bonsai, donde es solo una componente conexa completa, es decir, pudo conectar todas las imágenes en una sola pasada. Mientras que con bycicle y counter, en las menores fracciones son 2 componentes conexas, lo cual indicaría que no hubo suficiente matches validas para unir las fotos, haciendo que el modelo se partiera.

- Tiempo de ejecución: Esta métrica nuevamente se comporta similar entre las escenas. Se ve que el tiempo va creciendo exponencialmente en cada fracción, lo cual tiene sentido debido a que las comparaciones entre imagenes se hace de todas con todas, y aumentar la cantidad de imagenes es más trabajo.

- Track length promedio: Como explicamos anteriormente, esto es la cantidad de imagenes que aparece un punto 3D, en promedio. Y nuevamente podemos ver que el comportamiento es similar entre las 4 escenas, pero es mucho más robusto en el caso de garden y bonsai, donde a medida que aumenta la cantidad de imagenes, el track aumenta hasta a 7 camaras distintas, lo que indica que son puntos mucho más robustos.

Una conclusión final, es que las escenas, que logran una reconstrucción más robustas y de mayor calidad son garden y bonsai, en donde describimos en el experimento A, las imagenes tomadas son desde más ángulos y siempre (o casi siempre) enfocando en el centro el objeto.